In [0]:
from pyspark.sql.functions import col

In [0]:
%sql
CREATE OR REPLACE TABLE  brazilian_ecommerce.gold.fact_orders
TBLPROPERTIES (
  'config.nk_column_names'='',
  'config.cluster_by' = 'order_id,payment_type,order_status ',
  'config.fact_type' = 1,
  'config.sk_column_name' = 'orders_sk',
  'config.primary_key' = 'orders_sk',
  'config.foreign_key' = '',
  'config.unique_constraints' = '',
  'config.not_null_constraints' = '',
  'config.check_constraints' = ''
)
AS
select 
        md5(concat_ws('|', ord.order_id, pay.payment_sequential)) as orders_sk,  -- primary key
        ord.order_id,
        ord.customer_id,
        ord.order_status,
        cast(ord.order_purchase_timestamp as DATE) as order_purchase_timestamp,
        cast(ord.order_approved_at as DATE) as order_approved_at,
        cast(ord.order_delivered_carrier_date as DATE) as order_delivered_carrier_date,
        cast(ord.order_delivered_customer_date as DATE) as order_delivered_customer_date,
        cast(ord.order_estimated_delivery_date as DATE) as order_estimated_delivery_date,
        pay.payment_sequential,
        pay.payment_type,
        pay.payment_installments,
        sum(item.price) as price,
        sum(item.freight_value) as freight_value,
        sum(item.price) + sum(item.freight_value) as SalesAmount,
        sum(pay.payment_value) as payment_value
        from brazilian_ecommerce.silver.v_trn_oracle_ebs_olist_orders ord
        inner join brazilian_ecommerce.silver.v_trn_oracle_ebs_order_items item 
                on ord.order_id = item.order_id 
        inner join brazilian_ecommerce.silver.v_trn_oracle_ebs_olist_order_payments pay 
                on ord.order_id = pay.order_id
        group by all


In [0]:
df = spark.sql("""      
    select ifnull(payment_sequential, 100000) as payment_sequential
    from brazilian_ecommerce.silver.v_trn_oracle_ebs_olist_order_payments
""")

In [0]:
display(df.filter(col("payment_sequential")==100000))

In [0]:
df = spark.sql(""" 
               select 
        md5(concat_ws('|', ord.order_id, pay.payment_sequential)) as orders_sk,  -- primary key
        ord.order_id,
        ord.customer_id,
        ord.order_status,
        cast(ord.order_purchase_timestamp as DATE) as order_purchase_timestamp,
        cast(ord.order_approved_at as DATE) as order_approved_at,
        cast(ord.order_delivered_carrier_date as DATE) as order_delivered_carrier_date,
        cast(ord.order_delivered_customer_date as DATE) as order_delivered_customer_date,
        cast(ord.order_estimated_delivery_date as DATE) as order_estimated_delivery_date,
        pay.payment_sequential,
        pay.payment_type,
        pay.payment_installments,
        sum(item.price) as price,
        sum(item.freight_value) as freight_value,
        sum(item.price) + sum(item.freight_value) as SalesAmount,
        sum(pay.payment_value) as payment_value
        from brazilian_ecommerce.silver.v_trn_oracle_ebs_olist_orders ord
        inner join brazilian_ecommerce.silver.v_trn_oracle_ebs_order_items item 
                on ord.order_id = item.order_id 
        inner join brazilian_ecommerce.silver.v_trn_oracle_ebs_olist_order_payments pay 
                on ord.order_id = pay.order_id
        group by all
    """)


In [0]:
df.explain(extended=True)